<a href="https://colab.research.google.com/github/GiovanniMerici/Salmonella_Virulence/blob/main/spatial_genomics_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatial genomics analysis of discriminative *Salmonella* genes

This notebook maps discriminative genes onto representative *Salmonella* genomes, converts genomic proximity into adjacency networks, detects Leiden communities, and identifies communities shared across virulent or attenuated serovars.

The workflow corresponds to the spatial/localization analysis used for Figure 3 of the manuscript:

1. Split discriminative genes into virulent-enriched and attenuated-enriched FASTA files.
2. Download or use representative reference genomes for each serovar.
3. Run BLASTn of discriminative genes against representative genomes.
4. Keep best hits passing identity, coverage, and E-value thresholds.
5. Link genes located within 10 kb on the same contig.
6. Build serovar-specific adjacency networks.
7. Detect Leiden communities.
8. Compare communities across the three serovars in each group using Jaccard similarity.
9. Export tables and figures.

## Required local input files

Place these files in `data/` before running the notebook:

```text
virulence_conf.fa              # FASTA with all discriminative gene sequences
assigned_genes.pkl             # DataFrame with columns: gene, group_assignment
annotation_enterobase.tsv      # EnteroBase annotation table with Locus Tag and Description
```

The notebook creates outputs under `results/spatial_genomics/`.

## 1. Install dependencies

Run this cell in Google Colab. On an HPC/Conda environment, these packages should already be available through the project environment file.

In [ ]:
# Colab-only installation. Comment out if running in an existing Conda/HPC environment.
!apt-get update -qq
!apt-get install -y -qq ncbi-blast+
!pip install -q biopython pandas numpy networkx matplotlib matplotlib-venn python-igraph leidenalg

## 2. Configuration

Edit only this cell if your file locations or thresholds differ.

In [ ]:
from pathlib import Path

# Project folders
DATA_DIR = Path("data")
RESULTS_DIR = Path("results/spatial_genomics")
GENOME_DIR = DATA_DIR / "genomes"
QUERY_DIR = DATA_DIR / "queries"
BLAST_DIR = RESULTS_DIR / "blast"
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"

for directory in [DATA_DIR, RESULTS_DIR, GENOME_DIR, QUERY_DIR, BLAST_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Required input files
QUERY_FASTA = DATA_DIR / "virulence_conf.fa"
ASSIGNED_GENES_PKL = DATA_DIR / "assigned_genes.pkl"
ANNOTATION_TSV = DATA_DIR / "annotation_enterobase.tsv"

# NCBI Entrez email. Replace with your institutional email before downloading genomes.
ENTREZ_EMAIL = "giovanni.merici@unipr.it"

# BLAST and spatial-link thresholds used in the manuscript.
EVALUE = 1e-5
IDENTITY_THRESHOLD = 40.0
COVERAGE_THRESHOLD = 80.0
MAX_DISTANCE_KB = 10

# Community matching threshold across serovars.
JACCARD_THRESHOLD = 0.80

# Representative genomes used for the six serovars.
# Most are downloaded from NCBI nuccore. Kentucky is downloaded from the NCBI genome FTP path.
SEROVAR_REFERENCES = {
    "Typhimurium": {"source": "nuccore", "accession": "NC_016810.1"},
    "Monophasic": {"source": "nuccore", "accession": "LN999997.1"},
    "Enteritidis": {"source": "nuccore", "accession": "NC_011294.1"},
    "Derby": {"source": "nuccore", "accession": "CP029486.1"},
    "Rissen": {"source": "nuccore", "accession": "CP051213.1"},
    "Kentucky": {
        "source": "url_gzip",
        "url": "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/001/448/485/GCF_001448485.2_ASM144848v2/GCF_001448485.2_ASM144848v2_genomic.fna.gz",
    },
}

SEROVAR_GROUPS = {
    "virulent": ["Typhimurium", "Monophasic", "Enteritidis"],
    "attenuated": ["Rissen", "Derby", "Kentucky"],
}

## 3. Imports and input validation

In [ ]:
import gzip
import re
import shutil
import subprocess
import urllib.request
from collections import defaultdict

import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from Bio import Entrez, SeqIO
from matplotlib_venn import venn3, venn3_circles

Entrez.email = ENTREZ_EMAIL


def require_file(path: Path) -> None:
    """Stop early with a clear message if a required input file is missing."""
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}\n"
            "Place the file in the data/ folder or update the path in the configuration cell."
        )


for required in [QUERY_FASTA, ASSIGNED_GENES_PKL, ANNOTATION_TSV]:
    require_file(required)

print("All required input files found.")

## 4. Split the discriminative-gene FASTA by enrichment group

The FASTA is split using `assigned_genes.pkl`, which must contain the gene identifier and whether the gene is enriched in virulent or attenuated serovars.

In [ ]:
def base_gene_id(record_id: str) -> str:
    """Remove only a final numeric suffix, for example STMMW_12345_1 -> STMMW_12345."""
    return re.sub(r"_\d+$", "", str(record_id).split()[0])


def split_query_fasta_by_group(query_fasta: Path, assigned_genes_pkl: Path, output_dir: Path) -> dict[str, Path]:
    """Write one FASTA file for virulent-enriched genes and one for attenuated-enriched genes."""
    assigned = pd.read_pickle(assigned_genes_pkl)
    required_columns = {"gene", "group_assignment"}
    missing = required_columns - set(assigned.columns)
    if missing:
        raise ValueError(f"assigned_genes.pkl is missing columns: {sorted(missing)}")

    group_to_genes = {
        group: set(
            assigned.loc[assigned["group_assignment"].astype(str).str.lower() == group, "gene"]
            .astype(str)
            .str.strip()
        )
        for group in ["virulent", "attenuated"]
    }

    group_to_records = {"virulent": [], "attenuated": []}
    unmatched = []

    for record in SeqIO.parse(query_fasta, "fasta"):
        gene = base_gene_id(record.id)
        if gene in group_to_genes["virulent"]:
            group_to_records["virulent"].append(record)
        elif gene in group_to_genes["attenuated"]:
            group_to_records["attenuated"].append(record)
        else:
            unmatched.append((record.id, gene))

    output_paths = {}
    for group, records in group_to_records.items():
        out_fasta = output_dir / f"virulence_conf_{group}.fa"
        SeqIO.write(records, out_fasta, "fasta")
        output_paths[group] = out_fasta
        print(f"{group}: wrote {len(records)} records to {out_fasta}")

    print(f"unmatched records: {len(unmatched)}")
    return output_paths


QUERY_FASTAS_BY_GROUP = split_query_fasta_by_group(QUERY_FASTA, ASSIGNED_GENES_PKL, QUERY_DIR)

## 5. Download representative genomes

If the FASTA files already exist in `data/genomes/`, the notebook reuses them. This avoids unnecessary downloads on repeated runs.

In [ ]:
def download_nuccore_fasta(accession: str, output_fasta: Path) -> None:
    """Download one nucleotide FASTA record from NCBI nuccore using Entrez."""
    handle = Entrez.efetch(db="nuccore", id=accession, rettype="fasta", retmode="text")
    fasta_data = handle.read()
    output_fasta.write_text(fasta_data)


def download_gzipped_fasta(url: str, output_fasta: Path) -> None:
    """Download a gzipped FASTA from URL and decompress it."""
    tmp_gz = output_fasta.with_suffix(output_fasta.suffix + ".gz")
    urllib.request.urlretrieve(url, tmp_gz)
    with gzip.open(tmp_gz, "rt") as src, open(output_fasta, "w") as dst:
        shutil.copyfileobj(src, dst)
    tmp_gz.unlink(missing_ok=True)


def normalize_fasta_header(input_fasta: Path, output_fasta: Path, serovar: str) -> None:
    """Simplify FASTA headers so each representative genome is identified by serovar name."""
    records = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        record.id = serovar
        record.name = serovar
        record.description = serovar
        records.append(record)
    SeqIO.write(records, output_fasta, "fasta")


def get_reference_genome(serovar: str, info: dict, genome_dir: Path) -> Path:
    """Return local FASTA path for one serovar, downloading it if required."""
    final_fasta = genome_dir / f"{serovar}.fasta"
    if final_fasta.exists() and final_fasta.stat().st_size > 0:
        print(f"Using existing genome: {final_fasta}")
        return final_fasta

    raw_fasta = genome_dir / f"{serovar}.raw.fasta"
    print(f"Downloading {serovar} reference genome")

    if info["source"] == "nuccore":
        download_nuccore_fasta(info["accession"], raw_fasta)
    elif info["source"] == "url_gzip":
        download_gzipped_fasta(info["url"], raw_fasta)
    else:
        raise ValueError(f"Unsupported reference source for {serovar}: {info['source']}")

    normalize_fasta_header(raw_fasta, final_fasta, serovar)
    raw_fasta.unlink(missing_ok=True)
    return final_fasta


GENOME_FASTAS = {
    serovar: get_reference_genome(serovar, info, GENOME_DIR)
    for serovar, info in SEROVAR_REFERENCES.items()
}

## 6. Run BLASTn

This cell builds one BLAST database per representative genome and maps the group-specific discriminative genes to genomes from the same group.

In [ ]:
def run_command(command: list[str]) -> None:
    """Run an external command and raise a readable error if it fails."""
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            "Command failed:\n" + " ".join(command) +
            "\n\nSTDOUT:\n" + result.stdout +
            "\nSTDERR:\n" + result.stderr
        )


def make_blast_db(genome_fasta: Path, db_prefix: Path) -> None:
    """Create a nucleotide BLAST database if it is not already present."""
    if db_prefix.with_suffix(".nhr").exists() or Path(str(db_prefix) + ".nhr").exists():
        return
    run_command(["makeblastdb", "-in", str(genome_fasta), "-dbtype", "nucl", "-out", str(db_prefix)])


def run_blast_for_group(group: str) -> list[Path]:
    """Run BLASTn for all representative genomes in one virulence group."""
    query_fasta = QUERY_FASTAS_BY_GROUP[group]
    group_blast_dir = BLAST_DIR / group
    group_blast_dir.mkdir(parents=True, exist_ok=True)
    output_files = []

    for serovar in SEROVAR_GROUPS[group]:
        genome_fasta = GENOME_FASTAS[serovar]
        db_prefix = group_blast_dir / f"{serovar}_db"
        out_file = group_blast_dir / f"{serovar}.blast.tab"

        make_blast_db(genome_fasta, db_prefix)

        if out_file.exists() and out_file.stat().st_size > 0:
            print(f"Using existing BLAST output: {out_file}")
        else:
            print(f"Running BLAST: {group} genes against {serovar}")
            run_command([
                "blastn",
                "-query", str(query_fasta),
                "-db", str(db_prefix),
                "-out", str(out_file),
                "-outfmt", "6 qseqid sseqid pident length qlen slen qcovs qstart qend sstart send evalue bitscore",
                "-evalue", str(EVALUE),
                "-max_target_seqs", "1",
            ])
        output_files.append(out_file)

    return output_files


BLAST_OUTPUTS = {group: run_blast_for_group(group) for group in SEROVAR_GROUPS}

## 7. Parse and filter BLAST hits

For each query gene, only the best passing hit is retained. Thresholds are: E-value ≤ 1e-5, identity ≥ 40%, and coverage ≥ 80%.

In [ ]:
BLAST_COLUMNS = [
    "qseqid", "sseqid", "pident", "length", "qlen", "slen", "qcovs",
    "qstart", "qend", "sstart", "send", "evalue", "bitscore"
]


def filter_blast_hits(path: Path, identity_thresh: float, cov_thresh: float, evalue: float) -> pd.DataFrame:
    """Read one BLAST table, apply thresholds, and keep the best hit per query gene."""
    df = pd.read_csv(path, sep="	", header=None, names=BLAST_COLUMNS)
    if df.empty:
        return df

    df = df[
        (df["pident"] >= identity_thresh)
        & (df["qcovs"] >= cov_thresh)
        & (df["evalue"] <= evalue)
    ].copy()

    # Best hit per gene: lower E-value is better; if tied, higher bit score is better.
    df = df.sort_values(["qseqid", "evalue", "bitscore"], ascending=[True, True, False])
    df = df.drop_duplicates("qseqid", keep="first")

    # Convert BLAST coordinates into genomic intervals independent of strand.
    df["start"] = df[["sstart", "send"]].min(axis=1)
    df["end"] = df[["sstart", "send"]].max(axis=1)
    df["strand"] = np.where(df["sstart"] < df["send"], "+", "-")
    df["id"] = df["qseqid"].map(base_gene_id)
    return df


def parse_group_blast_outputs(group: str) -> tuple[dict[str, list[pd.DataFrame]], pd.DataFrame]:
    """Parse all BLAST outputs for one group and return filtered hits plus a recovery summary."""
    filtered_results = {}
    rows = []

    for serovar in SEROVAR_GROUPS[group]:
        blast_file = BLAST_DIR / group / f"{serovar}.blast.tab"
        df = filter_blast_hits(blast_file, IDENTITY_THRESHOLD, COVERAGE_THRESHOLD, EVALUE)
        filtered_results[serovar] = [df]
        rows.append({"group": group, "serovar": serovar, "genes_recovered": df["qseqid"].nunique()})

    summary = pd.DataFrame(rows)
    summary.to_csv(TABLE_DIR / f"{group}_blast_recovery_summary.csv", index=False)
    return filtered_results, summary


FILTERED_RESULTS = {}
BLAST_RECOVERY = []
for group in SEROVAR_GROUPS:
    FILTERED_RESULTS[group], summary = parse_group_blast_outputs(group)
    BLAST_RECOVERY.append(summary)

blast_recovery_summary = pd.concat(BLAST_RECOVERY, ignore_index=True)
display(blast_recovery_summary)

## 8. Build adjacency networks and Leiden communities

Genes are connected when they are consecutive on the same contig and separated by no more than 10 kb. Edge weights are the inverse of median inter-gene distance.

In [ ]:
def build_adjacent_edges(df: pd.DataFrame, genome_id: str, max_kb: int = 10) -> list[tuple[str, str, str, int]]:
    """Create edges between consecutive mapped genes on the same contig within max_kb."""
    edges = []
    for contig, sub in df.groupby("sseqid"):
        sub = sub.sort_values("start")
        rows = list(sub.itertuples(index=False))
        for idx in range(len(rows) - 1):
            left, right = rows[idx], rows[idx + 1]
            distance_bp = max(0, int(right.start) - int(left.end))
            if distance_bp > max_kb * 1000:
                continue
            if left.qseqid != right.qseqid:
                edges.append((left.qseqid, right.qseqid, genome_id, distance_bp))
    return edges


def build_serovar_edge_tables(filtered_results: dict[str, list[pd.DataFrame]], max_kb: int) -> dict[str, pd.DataFrame]:
    """Collapse adjacent-gene edges by serovar, keeping support and median distance."""
    serovar_edges = {}

    for serovar, dfs in filtered_results.items():
        raw_edges = []
        for genome_index, df in enumerate(dfs, start=1):
            genome_id = f"{serovar}_genome{genome_index}"
            raw_edges.extend(build_adjacent_edges(df, genome_id, max_kb=max_kb))

        edge_dict = defaultdict(lambda: {"genomes": set(), "distances": []})
        for gene_a, gene_b, genome_id, distance_bp in raw_edges:
            # Suffix the serovar name so homologous genes in different serovars remain separate nodes.
            node_a = f"{gene_a}|{serovar}"
            node_b = f"{gene_b}|{serovar}"
            key = tuple(sorted([node_a, node_b]))
            edge_dict[key]["genomes"].add(genome_id)
            edge_dict[key]["distances"].append(distance_bp)

        rows = []
        for (node_a, node_b), data in edge_dict.items():
            rows.append({
                "gene_u": node_a,
                "gene_v": node_b,
                "serovar": serovar,
                "support": len(data["genomes"]),
                "genomes": ";".join(sorted(data["genomes"])),
                "median_dist": float(np.median(data["distances"])),
            })

        serovar_edges[serovar] = pd.DataFrame(rows).sort_values(
            ["support", "median_dist"], ascending=[False, True]
        )

    return serovar_edges


def build_global_graph(serovar_edges: dict[str, pd.DataFrame]) -> nx.Graph:
    """Build a graph as the disjoint union of the serovar-specific adjacency graphs."""
    graph = nx.Graph()
    for serovar, edges_df in serovar_edges.items():
        for _, row in edges_df.iterrows():
            node_a = row["gene_u"]
            node_b = row["gene_v"]
            if node_a == node_b:
                continue
            median_dist = float(row["median_dist"])
            graph.add_edge(
                node_a,
                node_b,
                weight=1.0 / (1.0 + median_dist),
                dist=median_dist,
                support=int(row["support"]),
                serovar=serovar,
            )
    return graph


def run_leiden(graph: nx.Graph, seed: int = 42) -> dict[str, int]:
    """Run Leiden community detection and return node -> community mapping."""
    if graph.number_of_nodes() == 0:
        return {}
    graph_ig = ig.Graph.from_networkx(graph)
    partition = leidenalg.find_partition(
        graph_ig,
        leidenalg.RBConfigurationVertexPartition,
        weights="weight",
        seed=seed,
    )
    return {graph_ig.vs[i]["_nx_name"]: community for i, community in enumerate(partition.membership)}

## 9. Annotate communities

Descriptions are retrieved from the EnteroBase annotation table using the wgMLST locus tag.

In [ ]:
def load_annotation_table(annotation_tsv: Path) -> pd.DataFrame:
    """Load EnteroBase annotations and check the required columns."""
    annotation = pd.read_table(annotation_tsv)
    required = {"Locus Tag", "Description"}
    missing = required - set(annotation.columns)
    if missing:
        raise ValueError(f"Annotation table is missing columns: {sorted(missing)}")
    annotation["Locus Tag"] = annotation["Locus Tag"].astype(str)
    return annotation


def add_descriptions(community_df: pd.DataFrame, annotation: pd.DataFrame) -> pd.DataFrame:
    """Attach functional descriptions to community rows."""
    description_cache = {}

    def lookup_description(gene_id: str):
        if gene_id in description_cache:
            return description_cache[gene_id]
        matches = annotation.loc[annotation["Locus Tag"].str.contains(gene_id, regex=False), "Description"]
        value = matches.iloc[0] if not matches.empty else None
        description_cache[gene_id] = value
        return value

    annotated = community_df.copy()
    annotated["Description"] = annotated["id"].map(lookup_description)
    return annotated


ANNOTATION = load_annotation_table(ANNOTATION_TSV)

## 10. Compare communities across serovars

Communities are considered shared across serovars when their gene content has Jaccard similarity ≥ 0.80. Three-way shared communities require all three pairwise comparisons to meet the threshold; this avoids transitive matching artifacts.

In [ ]:
def jaccard(set_a: set, set_b: set) -> float:
    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 0.0


def identify_shared_communities(community_df: pd.DataFrame, serovars: list[str], threshold: float) -> tuple[pd.DataFrame, dict[str, set]]:
    """Identify unique, pairwise-shared, and three-way-shared community modules."""
    if len(serovars) != 3:
        raise ValueError("This function expects exactly three serovars.")

    comm_sets = community_df.groupby(["serovar", "community"])["id"].apply(lambda x: set(x.dropna()))
    A, B, C = serovars
    commA, commB, commC = comm_sets.loc[A], comm_sets.loc[B], comm_sets.loc[C]

    AB = {(cA, cB) for cA, gA in commA.items() for cB, gB in commB.items() if jaccard(gA, gB) >= threshold}
    AC = {(cA, cC) for cA, gA in commA.items() for cC, gC in commC.items() if jaccard(gA, gC) >= threshold}
    BC = {(cB, cC) for cB, gB in commB.items() for cC, gC in commC.items() if jaccard(gB, gC) >= threshold}

    A_to_C = defaultdict(set)
    B_to_C = defaultdict(set)
    for cA, cC in AC:
        A_to_C[cA].add(cC)
    for cB, cC in BC:
        B_to_C[cB].add(cC)

    triples = set()
    for cA, cB in AB:
        for cC in A_to_C.get(cA, set()) & B_to_C.get(cB, set()):
            triples.add((cA, cB, cC))

    A_in_ABC = {cA for cA, _, _ in triples}
    B_in_ABC = {cB for _, cB, _ in triples}
    C_in_ABC = {cC for _, _, cC in triples}

    modules_A = {("ABC", cA, cB, cC) for cA, cB, cC in triples}
    modules_B = set(modules_A)
    modules_C = set(modules_A)

    AB_only = {(cA, cB) for cA, cB in AB if cA not in A_in_ABC and cB not in B_in_ABC}
    AC_only = {(cA, cC) for cA, cC in AC if cA not in A_in_ABC and cC not in C_in_ABC}
    BC_only = {(cB, cC) for cB, cC in BC if cB not in B_in_ABC and cC not in C_in_ABC}

    modules_A |= {("AB", cA, cB) for cA, cB in AB_only} | {("AC", cA, cC) for cA, cC in AC_only}
    modules_B |= {("AB", cA, cB) for cA, cB in AB_only} | {("BC", cB, cC) for cB, cC in BC_only}
    modules_C |= {("AC", cA, cC) for cA, cC in AC_only} | {("BC", cB, cC) for cB, cC in BC_only}

    A_unique = set(commA.index) - {x for x, _ in AB} - {x for x, _ in AC}
    B_unique = set(commB.index) - {y for _, y in AB} - {x for x, _ in BC}
    C_unique = set(commC.index) - {y for _, y in AC} - {y for _, y in BC}

    modules_A |= {("A", cA) for cA in A_unique}
    modules_B |= {("B", cB) for cB in B_unique}
    modules_C |= {("C", cC) for cC in C_unique}

    shared_df = pd.DataFrame(triples, columns=[f"community_{A}", f"community_{B}", f"community_{C}"])
    module_sets = {A: modules_A, B: modules_B, C: modules_C}
    return shared_df, module_sets

## 11. Plotting helpers

In [ ]:
def plot_shared_community_venn(module_sets: dict[str, set], group: str, output_path: Path) -> None:
    """Plot Venn diagram of unique and shared community modules."""
    serovars = list(module_sets.keys())
    fill_colors = {
        "virulent": ["#f5b7b1", "#cb4335", "#922b21"],
        "attenuated": ["#6b8fd6", "#3f6fd1", "#1f3fa3"],
    }[group]

    plt.figure(figsize=(7, 6))
    venn = venn3([module_sets[s] for s in serovars], set_labels=serovars)

    for patch in venn.patches:
        if patch:
            patch.set_alpha(0.15)

    circles = venn3_circles([module_sets[s] for s in serovars], linestyle="solid", linewidth=3)
    for circle, color in zip(circles, fill_colors):
        if circle:
            circle.set_edgecolor(color)

    for label, color in zip(venn.set_labels, fill_colors):
        if label:
            label.set_color(color)
            label.set_fontsize(12)
            label.set_fontweight("bold")

    plt.title(f"Shared modules across {group} serovars", fontsize=13)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def plot_community_size_distribution(community_df: pd.DataFrame, group: str, output_path: Path) -> pd.DataFrame:
    """Plot the number of Leiden communities by community-size class for each serovar."""
    community_sizes = (
        community_df.groupby(["serovar", "community"]).size().reset_index(name="size")
    )

    def size_bin(n: int) -> str:
        if n == 2:
            return "2"
        if 3 <= n <= 5:
            return "3-5"
        if 6 <= n <= 10:
            return "6-10"
        return ">10"

    community_sizes["size_bin"] = community_sizes["size"].map(size_bin)
    summary = community_sizes.groupby(["serovar", "size_bin"], observed=False).size().unstack(fill_value=0)
    summary = summary.reindex(columns=["2", "3-5", "6-10", ">10"], fill_value=0)
    summary = summary.reindex(SEROVAR_GROUPS[group])

    colors = {
        "virulent": ["#fde0dd", "#fa9fb5", "#c51b8a", "#7a0177"],
        "attenuated": ["#eaf2ff", "#b6d0ff", "#5b8cff", "#355fdb"],
    }[group]

    ax = summary.plot(kind="bar", stacked=True, figsize=(8, 5), color=colors, edgecolor="white")
    ax.set_title(f"Distribution of community sizes in {group} serovars")
    ax.set_xlabel("Serovar")
    ax.set_ylabel("Number of communities")
    ax.legend(title="Community size (genes)", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    return summary


def plot_single_community(graph: nx.Graph, community_df: pd.DataFrame, community_id: int, output_path: Path, layout: str = "spring") -> None:
    """Plot one Leiden community with gene and description labels."""
    nodes = community_df.loc[community_df["community"] == community_id, "node"].tolist()
    nodes = [node for node in nodes if node in graph]
    subgraph = graph.subgraph(nodes).copy()
    if subgraph.number_of_nodes() == 0:
        raise ValueError(f"Community {community_id} contains no graph nodes.")

    if layout == "kamada":
        distances = {(u, v): subgraph[u][v].get("dist", 1.0) for u, v in subgraph.edges()}
        pos = nx.kamada_kawai_layout(subgraph, dist=distances)
    else:
        pos = nx.spring_layout(subgraph, seed=42, weight="weight")

    lookup = community_df.set_index("node")
    labels = {}
    for node in subgraph.nodes():
        row = lookup.loc[node]
        desc = row.get("Description", "")
        labels[node] = f"{row['gene']}\n{desc}" if pd.notna(desc) else row["gene"]

    widths = [subgraph[u][v].get("support", 1) for u, v in subgraph.edges()]
    plt.figure(figsize=(6, 6))
    nx.draw_networkx_edges(subgraph, pos, width=widths, alpha=0.4, edge_color="grey")
    nx.draw_networkx_nodes(subgraph, pos, node_color="royalblue", node_size=80, edgecolors="white", linewidths=0.4)
    nx.draw_networkx_labels(subgraph, pos, labels=labels, font_size=6)
    plt.axis("off")
    plt.title(f"Community {community_id}")
    plt.tight_layout()
    plt.savefig(output_path, dpi=600, bbox_inches="tight", transparent=True)
    plt.show()

## 12. Run the complete spatial-genomics workflow

This cell runs the analysis separately for virulent and attenuated serovars and writes all tables and figures.

In [ ]:
def run_spatial_workflow(group: str) -> dict:
    """Run adjacency construction, Leiden clustering, community sharing, and main plots for one group."""
    print(f"\n=== {group.upper()} SEROVARS ===")

    filtered_results = FILTERED_RESULTS[group]
    serovar_edges = build_serovar_edge_tables(filtered_results, max_kb=MAX_DISTANCE_KB)

    for serovar, edges_df in serovar_edges.items():
        edges_df.to_csv(TABLE_DIR / f"{group}_{serovar}_adjacency_edges.csv", index=False)

    graph = build_global_graph(serovar_edges)
    print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

    partition = run_leiden(graph, seed=42)
    community_df = pd.DataFrame({"node": list(partition.keys()), "community": list(partition.values())})
    community_df[["gene", "serovar"]] = community_df["node"].str.split("|", expand=True)
    community_df["id"] = community_df["gene"].map(base_gene_id)
    community_df = add_descriptions(community_df, ANNOTATION)
    community_df = community_df.sort_values(["community", "serovar", "gene"]).reset_index(drop=True)

    community_path = TABLE_DIR / f"{group}_leiden_communities.csv"
    community_df.to_csv(community_path, index=False)

    n_communities = community_df["community"].nunique()
    print(f"Leiden communities: {n_communities}")

    shared_df, module_sets = identify_shared_communities(
        community_df,
        serovars=SEROVAR_GROUPS[group],
        threshold=JACCARD_THRESHOLD,
    )
    shared_path = TABLE_DIR / f"{group}_shared_communities_jaccard_{JACCARD_THRESHOLD:.2f}.csv"
    shared_df.to_csv(shared_path, index=False)
    print(f"Three-way shared communities: {len(shared_df)}")

    size_summary = plot_community_size_distribution(
        community_df,
        group,
        FIGURE_DIR / f"{group}_community_size_distribution.png",
    )
    size_summary.to_csv(TABLE_DIR / f"{group}_community_size_distribution.csv")

    plot_shared_community_venn(
        module_sets,
        group,
        FIGURE_DIR / f"{group}_shared_communities_venn.png",
    )

    return {
        "serovar_edges": serovar_edges,
        "graph": graph,
        "partition": partition,
        "community_df": community_df,
        "shared_df": shared_df,
        "module_sets": module_sets,
        "size_summary": size_summary,
    }


SPATIAL_RESULTS = {group: run_spatial_workflow(group) for group in SEROVAR_GROUPS}

## 13. Optional: inspect or plot selected communities

Use this section only for targeted biological inspection, for example SPI-1, SPI-2, SPI-13, SPI-14, or arsenic-resistance communities. These cells are optional and do not change the saved output tables.

In [ ]:
# Example: list communities whose descriptions contain a keyword.
# Change GROUP and KEYWORD as needed.
GROUP = "virulent"
KEYWORD = "SPI"

community_df = SPATIAL_RESULTS[GROUP]["community_df"]
keyword_hits = community_df[
    community_df["Description"].fillna("").str.contains(KEYWORD, case=False, regex=False)
]

display(keyword_hits[["community", "serovar", "id", "Description"]].head(50))

In [ ]:
# Example: plot one community.
# Set COMMUNITY_ID to a value identified in the previous cell.
GROUP = "virulent"
COMMUNITY_ID = None  # for example: 3

if COMMUNITY_ID is not None:
    plot_single_community(
        SPATIAL_RESULTS[GROUP]["graph"],
        SPATIAL_RESULTS[GROUP]["community_df"],
        COMMUNITY_ID,
        FIGURE_DIR / f"{GROUP}_community_{COMMUNITY_ID}_network.png",
    )

## Output files

Main outputs are written to:

```text
results/spatial_genomics/tables/
results/spatial_genomics/figures/
```

Core tables:

- `virulent_leiden_communities.csv`
- `attenuated_leiden_communities.csv`
- `virulent_shared_communities_jaccard_0.80.csv`
- `attenuated_shared_communities_jaccard_0.80.csv`
- per-serovar adjacency edge tables

Core figures:

- `virulent_community_size_distribution.png`
- `attenuated_community_size_distribution.png`
- `virulent_shared_communities_venn.png`
- `attenuated_shared_communities_venn.png`